<a href="https://colab.research.google.com/github/priya25-art/Machine-Learning/blob/main/Linear%20Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [18]:
import pandas as pd

df = pd.read_excel("/content/ML_data.xlsx")
df.head()

,Student_ID,Department,Attendance,Internal_Marks,Study_Hours,Backlogs,Gender,Final_Result
0,101,CSE,85,78.0,12,0,Female,Pass
1,102,IT,62,45.0,5,2,Male,Fail
2,103,CSE,91,88.0,15,0,Female,Pass
3,104,ECE,74,NaN,8,1,Male,Pass
4,105,IT,55,39.0,4,3,Female,Fail


In [19]:
print("Dataset Info:\n", df.info())
print("\nMissing Values:\n", df.isnull().sum())
print("\nDuplicate Records:", df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Student_ID      5 non-null      int64  
 1   Department      5 non-null      object 
 2   Attendance      5 non-null      int64  
 3   Internal_Marks  4 non-null      float64
 4   Study_Hours     5 non-null      int64  
 5   Backlogs        5 non-null      int64  
 6   Gender          5 non-null      object 
 7   Final_Result    5 non-null      object 
dtypes: float64(1), int64(4), object(3)
memory usage: 452.0+ bytes
Dataset Info:
 None

Missing Values:
 Student_ID        0
Department        0
Attendance        0
Internal_Marks    1
Study_Hours       0
Backlogs          0
Gender            0
Final_Result      0
dtype: int64

Duplicate Records: 0


In [20]:
for col in df.select_dtypes(include=np.number).columns:
    df[col].fillna(df[col].median(), inplace=True)

/tmp/ipykernel_802/2155874100.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)


In [6]:
for col in df.select_dtypes(include="object").columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

/tmp/ipykernel_802/1936385350.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)


In [21]:
df=df.drop_duplicates()

In [22]:
import pandas as pd
import numpy as np

# 1. Load data
df = pd.read_excel("/content/ML_data.xlsx")

# 2. Impute missing values as done in earlier cells
for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna(df[col].mode()[0])

# 3. Drop duplicate records
df = df.drop_duplicates()

# 4. Separate features (X) and target (y)
# Excluding ID and target columns from features
X = df.drop(columns=["Student_ID", "Final_Result"])
y = df["Final_Result"]

# 5. Output inspection info
print(df.shape)
print(df.columns.tolist())
print("X shape:", X.shape, "| y shape:", y.shape)
print(y.describe())
print(X.dtypes)
display(X.head())

(5, 8)
['Student_ID', 'Department', 'Attendance', 'Internal_Marks', 'Study_Hours', 'Backlogs', 'Gender', 'Final_Result']
X shape: (5, 6) | y shape: (5,)
count        5
unique       2
top       Pass
freq         3
Name: Final_Result, dtype: object
Department         object
Attendance          int64
Internal_Marks    float64
Study_Hours         int64
Backlogs            int64
Gender             object
dtype: object


,Department,Attendance,Internal_Marks,Study_Hours,Backlogs,Gender
0,CSE,85,78.0,12,0,Female
1,IT,62,45.0,5,2,Male
2,CSE,91,88.0,15,0,Female
3,ECE,74,61.5,8,1,Male
4,IT,55,39.0,4,3,Female


In [23]:
categorical_cols = ["Department", "Gender"]

# Safely check if 'X' has been defined in the notebook session
if 'X' in globals():
    numeric_cols = [col for col in X.columns if col not in categorical_cols]
else:
    print("Warning: 'X' is not defined because the data could not be loaded.")
    print("Falling back to expected dataset schema columns.")
    # Default columns from the known dataset schema (excluding 'Student_ID', 'Final_Score', and categories)
    all_columns = ["Attendance_Pct", "Study_Hours_Per_Day", "Internal_Marks", "Assignments_Completed"]
    numeric_cols = [col for col in all_columns if col not in categorical_cols]

print("Categorical:", categorical_cols)
print("Numeric:", numeric_cols)

Categorical: ['Department', 'Gender']
Numeric: ['Attendance', 'Internal_Marks', 'Study_Hours', 'Backlogs']


In [24]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# 1. Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Define the preprocessor using the column groups identified earlier
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

# 3. Fit on training data and transform
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# See the shape
print("Processed X_train shape:", X_train_processed.shape)

# Get the resulting column names
feature_names = preprocessor.get_feature_names_out()
print("Feature names:", feature_names)

# View as a DataFrame
import pandas as pd
df_processed = pd.DataFrame(
    X_train_processed.toarray() if hasattr(X_train_processed, "toarray") else X_train_processed,
    columns=feature_names
)
df_processed.head()

Processed X_train shape: (4, 9)
Feature names: ['num__Attendance' 'num__Internal_Marks' 'num__Study_Hours'
 'num__Backlogs' 'cat__Department_CSE' 'cat__Department_ECE'
 'cat__Department_IT' 'cat__Gender_Female' 'cat__Gender_Male']


,num__Attendance,num__Internal_Marks,num__Study_Hours,num__Backlogs,cat__Department_CSE,cat__Department_ECE,cat__Department_IT,cat__Gender_Female,cat__Gender_Male
0,-1.551105,-1.489609,-1.386952,1.632993,0.0,0.0,1.0,1.0,0.0
1,1.076650,1.152594,1.266348,-0.816497,1.0,0.0,0.0,1.0,0.0
2,0.638690,0.613369,0.542720,-0.816497,1.0,0.0,0.0,1.0,0.0
3,-0.164235,-0.276353,-0.422116,0.000000,0.0,1.0,0.0,0.0,1.0


In [25]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import pandas as pd

# 1. Define a pipeline containing the preprocessor and a Logistic Regression classifier
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

# 2. Fit the pipeline on the training data
model.fit(X_train, y_train)

# 3. Retrieve feature names and coefficients
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
# For binary classification, coef_[0] represents the coefficients for the positive class
coefs = model.named_steps["classifier"].coef_[0]

# 4. Display sorted coefficients
pd.Series(coefs, index=feature_names).sort_values(ascending=False)

,0
num__Attendance,0.423493
num__Internal_Marks,0.392517
num__Study_Hours,0.344981
cat__Gender_Male,0.151462
cat__Department_ECE,0.151462
cat__Department_CSE,0.091152
cat__Gender_Female,-0.151376
cat__Department_IT,-0.242528
num__Backlogs,-0.470473


In [26]:
print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)
X_train.head()

(4, 6) (1, 6)
(4,) (1,)


,Department,Attendance,Internal_Marks,Study_Hours,Backlogs,Gender
4,IT,55,39.0,4,3,Female
2,CSE,91,88.0,15,0,Female
0,CSE,85,78.0,12,0,Female
3,ECE,74,61.5,8,1,Male


In [27]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Attendance',
                                                   'Internal_Marks',
                                                   'Study_Hours', 'Backlogs']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Department', 'Gender'])])),
                ('classifier', LogisticRegression())])

In [28]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Predict on the test set
y_pred = model.predict(X_test)

# Evaluate using classification metrics
print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, zero_division=0))

Accuracy Score: 1.0

Confusion Matrix:
 [[1]]

Classification Report:
               precision    recall  f1-score   support

        Fail       1.00      1.00      1.00         1

    accuracy                           1.00         1
   macro avg       1.00      1.00      1.00         1
weighted avg       1.00      1.00      1.00         1



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
